## Setting up the molecular Hamiltonian

In [1]:
# Force the local gqcpy to be imported
import sys
sys.path.insert(0, '../../build/gqcpy/')

import gqcpy

In [2]:
molecule = gqcpy.Molecule.ReadXYZ("../../gqcp/tests/data/h2_szabo.xyz" , 0)  # create a neutral molecule
N = molecule.numberOfElectrons()

In [3]:
spinor_basis = gqcpy.RSpinOrbitalBasis_cd(molecule, "STO-3G")

In [4]:
S = spinor_basis.quantize(gqcpy.OverlapOperator())
print(S.parameters())

[[0.99999999+0.j 0.65931816+0.j]
 [0.65931816+0.j 0.99999999+0.j]]


In [5]:
sq_hamiltonian = spinor_basis.quantize(gqcpy.FQMolecularHamiltonian(molecule))  # 'sq' for 'second-quantized'

In [6]:
print(sq_hamiltonian.core().parameters())

[[-1.12040896+0.j -0.95837989+0.j]
 [-0.95837989+0.j -1.12040896+0.j]]


In [7]:
print(sq_hamiltonian.twoElectron().parameters())

[[[[0.77460593+0.j 0.44410762+0.j]
   [0.44410762+0.j 0.56967589+0.j]]

  [[0.44410762+0.j 0.2970285 +0.j]
   [0.2970285 +0.j 0.44410762+0.j]]]


 [[[0.44410762+0.j 0.2970285 +0.j]
   [0.2970285 +0.j 0.44410762+0.j]]

  [[0.56967589+0.j 0.44410762+0.j]
   [0.44410762+0.j 0.77460593+0.j]]]]


In [8]:
environment = gqcpy.RHFSCFEnvironment_cd.WithCoreGuess(N, sq_hamiltonian, S)
solver = gqcpy.RHFSCFSolver_cd.DIIS()

In [9]:
objective = gqcpy.DiagonalRHFFockMatrixObjective_cd(sq_hamiltonian)  # use the default threshold of 1.0e-08

In [10]:
rhf_parameters = gqcpy.RHF_cd.optimize(objective, solver, environment).groundStateParameters()

In [11]:
C = rhf_parameters.expansion()
print(C.matrix())

[[-0.54893405-0.j -1.21146402+0.j]
 [-0.54893405-0.j  1.21146402+0.j]]


In [12]:
rhf_parameters.calculateStabilityMatrices(sq_hamiltonian)

In [13]:
rhf_parameters.orbitalSpace()

In [14]:
D = rhf_parameters.calculateScalarBasis1DM()
print(D.matrix())

[[0.60265718+0.j 0.60265718+0.j]
 [0.60265718+0.j 0.60265718+0.j]]


In [15]:
F = rhf_parameters.calculateScalarBasisFockMatrix(D, sq_hamiltonian)
print(F.parameters())

[[-0.36553732+0.j -0.59388534+0.j]
 [-0.59388534+0.j -0.36553732+0.j]]


In [16]:
h = sq_hamiltonian.core()
print(h.parameters())

[[-1.12040896+0.j -0.95837989+0.j]
 [-0.95837989+0.j -1.12040896+0.j]]


In [17]:
J = rhf_parameters.calculateScalarBasisDirectMatrix(D, sq_hamiltonian)
print(J.parameters())

[[1.34543038+0.j 0.89330201+0.j]
 [0.89330201+0.j 1.34543038+0.j]]


In [18]:
K = rhf_parameters.calculateScalarBasisExchangeMatrix(D, sq_hamiltonian)
print(K.parameters())

[[1.18111747+0.j 1.05761492+0.j]
 [1.05761492+0.j 1.18111747+0.j]]


In [19]:
F.parameters() == (h + J - 0.5*K).parameters()

array([[ True,  True],
       [ True,  True]])